# Exercise 03 — Capital allocation to risky assets

MSc Finance · Investments · FHNW · Autumn 2026

How much of your wealth belongs in the risky asset at all? This week the inputs are not
history but a forecast: the capital market assumptions a large asset manager publishes
every autumn and that pension funds across Europe feed into their strategic asset
allocation.

**How to work with this notebook.** The task text is here and on the exercise sheet. Each
code cell is a stub: the `# TODO` lines are the steps, in order. The setup and data cells
below are complete — run them first and leave them alone.

Run **Runtime → Restart and run all** before you trust any number in here.

**Data.** J.P. Morgan Asset Management, *2026 Long-Term Capital Market Assumptions*, 30th
annual edition, assumption matrix in **Swiss francs**, as of 30 September 2025. The file
holds Swiss cash and the complete equity block of the matrix. Source:
[am.jpmorgan.com/ch/en/asset-management/adv/insights/portfolio-insights/long-term-capital-market-assumptions](https://am.jpmorgan.com/ch/en/asset-management/adv/insights/portfolio-insights/long-term-capital-market-assumptions/).

Two columns of the matrix are returns, and the difference matters. The **arithmetic**
return is the input mean-variance analysis asks for; the **compound** return is what one
euro actually grows at, lower by roughly $\sigma^2/2$ — the gap you measured on historical
data in Exercise 02. Everything below uses the arithmetic column.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

from exercise_utils import FHNW, setup_style, save_results
setup_style()

pct = "{:.2%}".format

In [ ]:
BASE = "https://raw.githubusercontent.com/KroeTiA/Investments/main/"
DATA_URL = BASE + "Exercise_03/data/ltcma_2026_chf_equities.csv"

# Four lines of provenance sit on top of the file. All figures are in percent per annum.
cma = pd.read_csv(DATA_URL, skiprows=4).set_index("Asset class")
cma = cma.rename(columns={"Arithmetic return 2026 (%)": "ER",
                          "Annualized volatility (%)": "SD",
                          "Compound return 2026 (%)": "Compound"})
cma[["ER", "SD", "Compound"]] /= 100.0

RF = cma.loc["Swiss Cash", "ER"]                    # Swiss cash — the risk-free asset
menu = cma.loc[cma["Category"] == "Equity", ["ER", "SD"]].copy()

P = "AC World Equity"                               # the pre-specified risky portfolio
ER_P, SD_P = menu.loc[P, "ER"], menu.loc[P, "SD"]

print(f"risk-free rate {RF:.2%}          {len(menu)} equity asset classes")
print(f"{P}:  E(R) {ER_P:.2%}   sigma {SD_P:.2%}   risk premium {ER_P - RF:.2%}")

## Task 1 — Scoring the menu

Two clients walk in. Roger has risk aversion $A = 2$, Martina has $A = 6$. Neither of them
holds cash: whatever they choose, they are fully invested in one equity asset class.

Score every asset class on the menu with the mean-variance utility function

$$U = E(R) - \tfrac{1}{2} A \sigma^2$$

and rank the seventeen asset classes for each client. Report the top three and the bottom
three for both.

Then find the risk aversion at which the two clients would agree. Take **Swiss Equity** and
**AC World Equity** — the home market and the global one — and solve for the $A$ at which
they are equally attractive. State which of the two an investor prefers above that value
and which below.

Finally, draw the menu in $(\sigma, E(R))$ space and add the indifference curve running
through the winning asset for each client. Choose the axis ranges yourself.

*Deliverable: the ranking table for both clients, the break-even $A$ with one sentence on
which side is which, and the figure with two indifference curves.*

In [ ]:
# SOLUTION
# The utility score of an asset with mean er and volatility sd.             # KEEP
def utility(er, sd, A):                                                       # KEEP
    # STUB: one line — the utility function from the lecture
    return er - 0.5 * A * sd**2


scores = menu.copy()                                                          # KEEP

# STUB: a utility column and a rank column for each of the two clients
for A in (2, 6):
    scores[f"U (A={A})"] = utility(scores["ER"], scores["SD"], A)
    scores[f"Rank (A={A})"] = scores[f"U (A={A})"].rank(ascending=False).astype(int)

print(scores.sort_values("Rank (A=2)")                                        # KEEP
            .to_string(formatters={c: pct for c in scores.columns             # KEEP
                                   if not c.startswith("Rank")}))             # KEEP

In [ ]:
# SOLUTION
home, world = "Swiss Equity", "AC World Equity"                               # KEEP

# STUB: the A at which the two assets deliver the same utility score
A_star = (2 * (menu.loc[world, "ER"] - menu.loc[home, "ER"])
          / (menu.loc[world, "SD"]**2 - menu.loc[home, "SD"]**2))

print(f"break-even risk aversion  A* = {A_star:.2f}")
for A in (A_star - 1, A_star + 1):
    u_home = utility(menu.loc[home, "ER"], menu.loc[home, "SD"], A)
    u_world = utility(menu.loc[world, "ER"], menu.loc[world, "SD"], A)
    print(f"  A = {A:.2f}:  {home} {u_home:7.2%}   {world} {u_world:7.2%}"
          f"   -> {home if u_home > u_world else world}")

In [ ]:
# SOLUTION
grid = np.linspace(0.0, 0.32, 300)                                            # KEEP
LABEL = ["Swiss Equity", "AC World Equity", "Japanese Equity",                # KEEP
         "Chinese Domestic Equity", "Euro Area Small Cap", "U.S. Small Cap"]  # KEEP

fig1, ax = plt.subplots(figsize=(8.4, 5.0))                                   # KEEP
ax.scatter(menu["SD"], menu["ER"], s=28, color=FHNW["navy"], zorder=3)        # KEEP
for name in LABEL:                                                            # KEEP
    ax.annotate(name, (menu.loc[name, "SD"], menu.loc[name, "ER"]),           # KEEP
                fontsize=8, xytext=(6, -4), textcoords="offset points")       # KEEP

for A, colour in [(2, FHNW["blue"]), (6, FHNW["orange"])]:                    # KEEP
    winner = scores[f"U (A={A})"].idxmax()

    # STUB: the indifference curve through the winner, E(R) = U + 0.5 * A * sigma^2
    U_win = scores.loc[winner, f"U (A={A})"]
    ax.plot(grid, U_win + 0.5 * A * grid**2, color=colour, lw=1.6,
            label=f"A = {A}, through {winner}")

ax.set_xlabel("Volatility (annual)")                                          # KEEP
ax.set_ylabel("Expected return (annual, arithmetic)")                         # KEEP
ax.set_xlim(0.10, 0.32)                                                       # KEEP
ax.set_ylim(0.02, 0.13)                                                       # KEEP
ax.xaxis.set_major_formatter(PercentFormatter(1.0))                           # KEEP
ax.yaxis.set_major_formatter(PercentFormatter(1.0))                           # KEEP
ax.legend(loc="upper left")                                                   # KEEP
plt.show()                                                                    # KEEP

<!-- solution -->
**Roger ($A = 2$).** Japanese Equity 6.91 %, Euro Area Small Cap 5.60 %, AC Asia ex-Japan
Equity 5.48 % at the top; U.S. Small Cap 3.80 %, Swiss Equity 3.37 % and Chinese Domestic
Equity 2.25 % at the bottom.

**Martina ($A = 6$).** Japanese Equity 2.24 %, then AC World Equity $-0.01\ \%$ and AC World
ex-EMU Equity $-0.03\ \%$. The bottom three are UK Small Cap, U.S. Small Cap and Chinese
Domestic Equity.

Two things in that comparison are worth naming. The first is how far the ranking moves:
Swiss Equity climbs from 16th to 4th, Euro Area Small Cap falls from 2nd to 12th. Nothing
about the assets changed — only the price the client puts on variance. The second is the
level. At $A = 6$ almost every score is **negative**, and negative here means worse than
holding nothing at all: a utility score is measured in the same units as a return, so
Martina's $-1.22\ \%$ on U.S. Large Cap says she would rather have a certain $-1.22\ \%$
than that asset. Cash at 1.20 % beats every single line of her table. That is not a
statement about equities; it is a statement about the constraint "fully invested", which
Task 2 removes.

Chinese Domestic Equity is the clean illustration of why a high expected return is not an
argument. It has the highest $E(R)$ on the menu at 10.75 % and finishes last for both
clients, because 29.16 % volatility costs $\tfrac{1}{2} A \sigma^2 = 8.5$ percentage points
even for Roger.

**The break-even.** $A^* = 6.41$. Above it the home market wins, below it the global one.
Note what the formula

$$A^* = \frac{2\,[E(R_W) - E(R_H)]}{\sigma_W^2 - \sigma_H^2}$$

does *not* contain: the levels. Only the two differences matter, which is why the answer
survives adding a constant to both expected returns. Swiss Equity is the lower-return,
lower-risk asset, so it wins only for the sufficiently risk-averse — and $A = 6.41$ is a
high bar, roughly the risk aversion a pension fund reveals.

The mistake to expect in the figure: plotting the indifference curve as a straight line.
The utility function is linear in $\sigma^2$ but the axis is $\sigma$, so the curve is a
parabola opening upwards, and it gets steeper for larger $A$. A group that draws a line has
plotted $U = E(R) - \tfrac{1}{2} A \sigma$ and should notice that the curve then passes
through the wrong asset.

## Task 2 — Adding cash changes the question

Now let both clients hold Swiss cash alongside the risky asset. A share $y$ goes into the
risky asset and $1 - y$ into cash, so the complete portfolio has

$$E(R_C) = R_f + y\,[E(R_P) - R_f], \qquad \sigma_C = y\,\sigma_P .$$

Write a function `utility_of_y(y, er, sd, A, rf)` that returns the utility score of the
complete portfolio. Evaluate it over a grid of $y$ from 0 to 2 for the pre-specified risky
portfolio $P$ = AC World Equity, once for each client, and plot the two curves. Mark the
maximum you read off the grid and check it against the closed form

$$y^* = \frac{E(R_P) - R_f}{A\,\sigma_P^2}.$$

Then go back to the full menu. For every asset class compute the Sharpe ratio, the optimal
share $y^*$ and the utility at the optimum $U^* = U(y^*)$, for both clients, and rank the
menu by $U^*$.

Compare that ranking with the two rankings from Task 1. **Say what changed, and explain
why.** One number in your table explains the whole thing.

*Deliverable: the $U(y)$ figure with four marked points, the menu table with $SR$, $y^*$ and
$U^*$ for both clients, and two or three sentences on why the ranking behaves as it does.*

In [ ]:
# SOLUTION
# Utility of a complete portfolio holding a share y of the risky asset.      # KEEP
def utility_of_y(y, er, sd, A, rf):                                           # KEEP
    # STUB: the mean and the volatility of the complete portfolio, then reuse utility()
    return utility(rf + y * (er - rf), y * sd, A)


ys = np.linspace(0, 2, 401)                                                   # KEEP
fig2, ax = plt.subplots(figsize=(8.4, 4.6))                                   # KEEP

for A, colour in [(2, FHNW["blue"]), (6, FHNW["orange"])]:                    # KEEP
    # STUB: the utility curve, the grid maximum, and the closed-form y*
    u = utility_of_y(ys, ER_P, SD_P, A, RF)
    y_grid = ys[np.argmax(u)]
    y_star = (ER_P - RF) / (A * SD_P**2)

    ax.plot(ys, u, color=colour, lw=1.8, label=f"A = {A}")                    # KEEP
    ax.scatter([y_grid], [u.max()], s=45, color=colour, zorder=3)             # KEEP
    print(f"A = {A}:  grid {y_grid:.3f}   closed form {y_star:.3f}   "
          f"U* {utility_of_y(y_star, ER_P, SD_P, A, RF):.2%}")

ax.axhline(RF, color="#999999", lw=1.0, ls="--")                              # KEEP
ax.set_xlabel("Share y in AC World Equity")                                   # KEEP
ax.set_ylabel("Utility score of the complete portfolio")                      # KEEP
ax.legend()                                                                   # KEEP
plt.show()                                                                    # KEEP

In [ ]:
# SOLUTION
opt = menu.copy()                                                             # KEEP

# STUB: the Sharpe ratio of each asset class
opt["SR"] = (opt["ER"] - RF) / opt["SD"]

# STUB: y* and U* for each client, then rank the menu by U*
for A in (2, 6):
    opt[f"y* (A={A})"] = (opt["ER"] - RF) / (A * opt["SD"]**2)
    opt[f"U* (A={A})"] = utility_of_y(opt[f"y* (A={A})"], opt["ER"], opt["SD"], A, RF)
    opt[f"Rank (A={A})"] = opt[f"U* (A={A})"].rank(ascending=False).astype(int)

print(opt.sort_values("Rank (A=2)").to_string(                                # KEEP
    formatters={"ER": pct, "SD": pct, "SR": "{:.3f}".format,                   # KEEP
                "y* (A=2)": "{:.2f}".format, "y* (A=6)": "{:.2f}".format,      # KEEP
                "U* (A=2)": pct, "U* (A=6)": pct}))                            # KEEP

# STUB: check the two U* rankings against each other and against the SR ranking
print(f"\nthe two U* rankings agree everywhere: "
      f"{opt['Rank (A=2)'].equals(opt['Rank (A=6)'])}")
print(f"and they agree with the Sharpe-ratio ranking: "
      f"{opt['Rank (A=2)'].equals(opt['SR'].rank(ascending=False).astype(int))}")

<!-- solution -->
The grid and the closed form agree to the resolution of the grid: $y^* = 1.261$ for Roger
and $0.420$ for Martina, both recovered as 1.260 and 0.420 from 401 grid points. Roger
borrows a quarter of his wealth at the Swiss cash rate; Martina holds 58 % in cash.

Now the table, and the result the task is built around. **The $U^*$ ranking is identical for
the two clients, and identical to the Sharpe-ratio ranking.** Both lines of the check print
`True`. Risk aversion decides *how much* risk to take; it has nothing to say about *which*
risky asset to take. That is the separation property, arriving two lectures before we prove
it, and it is why Lecture 05 will maximise a Sharpe ratio rather than a utility score —
maximising utility over the choice of $P$ would give an answer that depends on $A$, and it
does not.

The algebra behind it is two lines. Substituting $y^*$ back into $U(y)$,

$$U^* = R_f + \frac{[E(R_P) - R_f]^2}{2 A \sigma_P^2} = R_f + \frac{SR^2}{2A}.$$

$A$ enters as a divisor of the *whole* second term, so it scales every asset's score by the
same factor and cannot reorder them. Check it on any row of the table: Japanese Equity has
$SR = 0.526$, so $U^*(A=6) = 1.20\ \% + 0.526^2/12 = 3.51\ \%$, which is what the column
says.

Three consequences worth having in your notes.

*Swiss Equity is last.* Ranked 3rd for Martina in Task 1, it is 17th of 17 here — the worst
asset class on the menu for both clients. Low volatility was an advantage only while cash
was forbidden. Once you can hold cash, you can de-risk *any* asset by lowering $y$, so
volatility stops being a disqualification and only the compensation per unit of it counts.
Swiss Equity offers 3.93 % of risk premium for 13.28 % of volatility, and that trade is the
poorest on offer.

*Chinese Domestic Equity beats it.* Last for both clients in Task 1, it finishes 16th
here — ahead of Swiss Equity. The same mechanism read from the other end: 29 % volatility
is not an objection when you are free to hold only 19 % of it.

*Every $U^*$ exceeds $R_f$.* Martina's scores were negative throughout Task 1 and are all
above 1.20 % here. $y = 0$ is always available, so the optimum can never be worse than cash.

The predictable error is a `utility_of_y` that computes $\sigma_C = y^2 \sigma_P$ or
forgets that $(1-y)$ earns $R_f$ rather than nothing. Both produce curves with a maximum in
roughly the right place, which is why the closed-form check is in the task: it is the only
thing that catches them.

## Task 3 — How much does it cost to be wrong?

Capital market assumptions are forecasts, and J.P. Morgan revises them every autumn. Stay
with $P$ = AC World Equity and take a client with $A = 3$.

First, how far does the answer move when the input moves? Plot $y^*$ against $E(R_P)$ over
$\pm 2$ percentage points, and against $\sigma_P$ over $\pm 4$ percentage points, in two
panels. Then put a number on each: compute the elasticity of $y^*$ with respect to the risk
premium and with respect to volatility, either from the formula or numerically. The two
numbers are small integers.

Second, how much does the error cost? The loss in utility from holding $y$ instead of $y^*$
is a quadratic,

$$U(y^*) - U(y) = \tfrac{1}{2} A \sigma_P^2 (y - y^*)^2 ,$$

and a utility score is measured in return units, so the loss is readable in basis points of
certainty-equivalent return. Plot $U(y)$ for $A = 3$ and shade the range of $y$ that costs
less than 10 basis points. Report the width of that range in percentage points, and the cost
of two specific mistakes: holding $y = 1$ instead of $y^*$, and keeping $y^*$ unchanged
after a revision that raises $\sigma_P$ from 15.87 % to 17.87 %.

*Deliverable: the two-panel sensitivity figure with the two elasticities, the no-regret
width in percentage points, and the two costs in basis points.*

In [ ]:
# SOLUTION
A = 3                                                                         # KEEP
y_star_P = (ER_P - RF) / (A * SD_P**2)                                        # KEEP

fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.0, 4.0))                    # KEEP
ers = np.linspace(ER_P - 0.02, ER_P + 0.02, 200)                              # KEEP
sds = np.linspace(SD_P - 0.04, SD_P + 0.04, 200)                              # KEEP

# STUB: y* as the expected return varies, then as the volatility varies
ax1.plot(ers, (ers - RF) / (A * SD_P**2), color=FHNW["blue"], lw=1.8)
ax2.plot(sds, (ER_P - RF) / (A * sds**2), color=FHNW["blue"], lw=1.8)

panels = [(ax1, ER_P, "Expected return of P"), (ax2, SD_P, "Volatility of P")]  # KEEP
for ax, x0, xlab in panels:                                                   # KEEP
    ax.scatter([x0], [y_star_P], s=45, color=FHNW["navy"], zorder=3)          # KEEP
    ax.set_xlabel(xlab)                                                       # KEEP
    ax.set_ylabel("Optimal share y*")                                         # KEEP
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))                       # KEEP

fig3.tight_layout()                                                           # KEEP
plt.show()                                                                    # KEEP

# STUB: the two elasticities, d log y* / d log x, at the published inputs
y_of = lambda er, sd: (er - RF) / (A * sd**2)
h = 1e-5
el_rp = ((y_of(ER_P + h, SD_P) / y_star_P - 1) / (h / (ER_P - RF)))
el_sd = ((y_of(ER_P, SD_P + h) / y_star_P - 1) / (h / SD_P))

print(f"y* at the published inputs                {y_star_P:.3f}")
print(f"elasticity of y* w.r.t. the risk premium  {el_rp:+.2f}")
print(f"elasticity of y* w.r.t. volatility        {el_sd:+.2f}")

In [ ]:
# SOLUTION
# STUB: the curvature of the utility function in y, and the loss from holding y
curv = 0.5 * A * SD_P**2
loss = lambda y: curv * (y - y_star_P)**2

# STUB: the half-width of the range that costs less than 10 basis points
half = np.sqrt(0.0010 / curv)

fig4, ax = plt.subplots(figsize=(8.4, 4.4))                                   # KEEP
ax.plot(ys, utility_of_y(ys, ER_P, SD_P, A, RF), color=FHNW["navy"], lw=1.8)  # KEEP
ax.axvspan(y_star_P - half, y_star_P + half, color=FHNW["green"], alpha=0.15) # KEEP
ax.axvline(y_star_P, color=FHNW["green"], lw=1.2)                             # KEEP
ax.set_xlabel("Share y in AC World Equity")                                   # KEEP
ax.set_ylabel("Utility score of the complete portfolio")                      # KEEP
plt.show()                                                                    # KEEP

print(f"y* = {y_star_P:.3f}, and anything within +/-{100*half:.1f} pp of it "
      f"costs less than 10 bp")
print(f"cost of holding y = 1.00 instead:      {10000*loss(1.0):5.1f} bp")

# STUB: after the revision, the new y* and the cost of sticking with the old one
SD_NEW = 0.1787
y_new = (ER_P - RF) / (A * SD_NEW**2)
stale = 0.5 * A * SD_NEW**2 * (y_star_P - y_new)**2
print(f"revision sigma 15.87% -> 17.87%: y* falls {y_star_P:.3f} -> {y_new:.3f} "
      f"({100*(y_new/y_star_P - 1):+.1f} %)")
print(f"cost of not acting on the revision:    {10000*stale:5.1f} bp")

<!-- solution -->
The elasticities are $+1$ for the risk premium and $-2$ for volatility, and they are visible
in the figure: the left panel is a straight line, the right one a convex decay. They come
straight off $y^* = [E(R_P) - R_f] / (A\sigma_P^2)$ — the premium appears once in the
numerator, volatility twice in the denominator. In practical terms a 10 % error in the
volatility forecast moves the recommended equity share by 20 %, twice as much as a 10 %
error in the risk premium. The lecture's comparative statics said "linear in the premium,
inverse in the variance"; this is that sentence with a number attached.

So the allocation is a fragile object. The second half of the task is the correction to
that impression.

$y^* = 0.840$ at $A = 3$. The utility loss from missing it is quadratic with curvature
$\tfrac{1}{2}A\sigma_P^2 = 0.0378$, which means **anything between 68 % and 100 % equity
costs less than 10 basis points** of certainty-equivalent return — a band 32 percentage
points wide. Holding a round $y = 1.00$ instead of the optimal 0.840 costs 9.6 bp. And the
$\sigma$ revision that moved $y^*$ by 21 %, from 0.840 to 0.663, costs 15.1 bp per year if
you ignore it entirely.

Put the two halves together and you have the practical result: **$y^*$ is a sensitive
function of inputs you cannot forecast well, and precisely because the objective is flat
near its maximum, that sensitivity does not matter much.** This is why an investment
committee can argue for an afternoon about 60 % versus 65 % equities and why the argument is
worth less than it feels. It is also the first appearance of a theme that returns in Lecture
05 with much sharper teeth: there, the optimiser sits on a surface that is flat in some
directions and the input errors are correlated across assets, and the two effects no longer
cancel so kindly.

Read the quadratic once more before moving on: the loss depends on $(y - y^*)^2$ but also on
$A\sigma_P^2$. A more risk-averse client, or a more volatile asset, has a *narrower*
no-regret band. Flatness is not a free property of the problem; it is a property of these
inputs.

The common error is to compute the cost of the revision as the loss under the *old*
volatility. The revision changes the world, not just the recommendation: the loss has to be
evaluated at $\sigma = 17.87\ \%$, which is what makes it 15.1 bp rather than 11.8 bp.

## Task 4 — Optional: what borrowing actually costs

Roger's $y^* = 1.26$ assumed he can borrow at the Swiss cash rate. He cannot. Suppose the
margin rate is $R_B = R_f + 1.5\ \%$, so the capital allocation line has a kink at $y = 1$:
lending below it, borrowing above it.

Compute the optimal $y$ under the kinked line for risk aversions from 1 to 8 and plot it
against $A$, with the unconstrained $y^*$ for comparison. There is an interval of $A$ over
which the answer is exactly $y = 1$. Find its endpoints and say in one sentence why an
interval appears at all.

*Deliverable: the figure, the two endpoints of the interval, and the sentence.*

In [ ]:
# SOLUTION
SPREAD = 0.015                                                                # KEEP
As = np.linspace(1, 8, 200)                                                   # KEEP

# STUB: y* using the lending rate, y* using the borrowing rate, then pick the
# STUB: one that is feasible on its own side of the kink; otherwise the corner y = 1
y_lend = (ER_P - RF) / (As * SD_P**2)
y_borrow = (ER_P - RF - SPREAD) / (As * SD_P**2)
y_kinked = np.where(y_lend <= 1, y_lend, np.where(y_borrow >= 1, y_borrow, 1.0))

fig5, ax = plt.subplots(figsize=(8.4, 4.4))                                   # KEEP
ax.plot(As, y_lend, color="#999999", lw=1.4, ls="--",                         # KEEP
        label="no borrowing spread")                                          # KEEP
ax.plot(As, y_kinked, color=FHNW["navy"], lw=2.0,                             # KEEP
        label=f"borrowing at Rf + {SPREAD:.1%}")                              # KEEP
ax.axhline(1.0, color=FHNW["red"], lw=1.0, ls=":")                            # KEEP
ax.set_xlabel("Risk aversion A")                                              # KEEP
ax.set_ylabel("Optimal share y")                                              # KEEP
ax.legend()                                                                   # KEEP
plt.show()                                                                    # KEEP

# STUB: the two risk aversions at which the corner starts and ends
A_low = (ER_P - RF - SPREAD) / SD_P**2
A_high = (ER_P - RF) / SD_P**2
print(f"y = 1 exactly for A between {A_low:.2f} and {A_high:.2f}")

<!-- solution -->
The corner runs from $A = 1.93$ to $A = 2.52$. Below 1.91 the client borrows even at the
higher rate; above 2.52 she lends; in between she does neither and holds exactly the risky
portfolio.

The interval appears because the objective has a kink, not a jump. To the left of $y = 1$
the marginal benefit of more equity is priced off $R_f$; to the right it is priced off
$R_B$. For $A$ between the two endpoints the first derivative is positive approaching $y=1$
from below and negative approaching it from above, so the maximum sits exactly at the kink
and stays there over a range of $A$. A single rate would give one $A$ per allocation; a
spread gives a whole interval of investors the same answer.

Worth noticing: the endpoints are $[E(R_P) - R_f - \text{spread}] / \sigma_P^2$ and
$[E(R_P) - R_f] / \sigma_P^2$, so the width of the corner is $\text{spread}/\sigma_P^2$. A
wider spread, or a less volatile risky portfolio, flattens more investors onto $y = 1$ —
which is a large part of why "fully invested, no leverage" is the default in practice
rather than an assumption anyone had to defend.

## Export

Bundle the figures and the tables, in case you want them for the transfer questions or your
own notes.

In [ ]:
# SOLUTION
# STUB: export the two tables together with the figures you want to keep
save_results(figures={"menu_indifference": fig1, "utility_of_y": fig2,
                      "sensitivity": fig3},
             tables={"scores": scores, "optimal": opt},
             name="ex03")

## Where this goes next

Two quizzes are open in Moodle until Sunday: the cumulative drill and the transfer
questions. Both are ungraded, and both are exactly the format the exams use.

Task 2 left one question standing. The ranking by $U^*$ said the whole menu should be held
in whichever single asset class has the highest Sharpe ratio, and no investor anywhere does
that. Lecture 04 supplies the missing ingredient: the assets on this menu move together
only partly, and a combination of them can have a higher Sharpe ratio than any of its
members.